# NB89 - P8 ablation row: `clr_lane_no_square_priors`

**Row 3** of the P8 ablation. Base `CLRHead` with CULane's 24/144/24
prior split (instead of the square-image-tuned 32/128/32). Headline
question: does the prior retuning matter, or is the CULane default fine?

**Self-contained**: this notebook mounts Drive, extracts the full BDD
dataset to `/content/`, and launches a SINGLE training run. The other
two P8 ablation rows are in NB{88, 89, 90} \ this one - run them on
separate Colab runtimes for parallelism. NB91 aggregates the 3 results
into the final ablation table.

**Per-epoch saves**: `last.pt`, `best.pt` (only when val improves), and
a `log_epoch_NNN.json` metrics snapshot are mirrored to
`/content/drive/MyDrive/EcoCAR/training_runs/checkpoints/clr_lane_no_square_priors/`
at the end of every epoch. Safe against Colab disconnects.

**Per-epoch metrics table**: validation runs after every epoch and a
fixed-width row of detection + lane metrics is printed below the loss
columns.


### Cell 1: Mount Drive + locate sources

In [1]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

os.environ['PYTHONIOENCODING'] = 'utf-8'

if not Path('/content/drive').exists():
    drive.mount('/content/drive', force_remount=False)

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import mmcv  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
print('[ok] env ready')


Mounted at /content/drive
[ok] env ready


### Cell 2: Extract FULL BDD dataset to /content/

70k train + 10k val images + matching detection labels + matching
(max_lanes, 78) polyline targets. Same prep script the smoke / brief
notebooks use, just with `n_train=70000, n_val=10000`.

The dataset goes on `/content/` (Colab's local SSD) - NEVER on Drive,
per the project's Drive-vs-local rule. Each Colab runtime extracts its
own copy.


In [2]:
import sys, subprocess
from pathlib import Path

PREP_SCRIPT = Path('stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py')
SUBSET_ROOT = Path('/content/bdd_dataset')

# Pre-flight: missing prep script or missing Drive inputs are the most
# common causes of "exit 2" with no visible stderr (subprocess.check_call
# in Colab can swallow the child's stderr on argparse / "file not found"
# style failures). Surface them explicitly before launching.
if not PREP_SCRIPT.exists():
    raise FileNotFoundError(
        f'prep script not found relative to CWD={Path.cwd()}:\n  {PREP_SCRIPT}\n'
        'Verify the Drive copy of yolop_vehicle_lane is up to date.'
    )
required_inputs = [
    Path('/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'),
    Path('/content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip'),
    Path('/content/drive/MyDrive/EcoCAR/datasets/lane_targets_clr_v1_polyline.tar.gz'),
]
missing = [str(p) for p in required_inputs if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Required Drive input(s) missing - verify these exist:\n  '
        + '\n  '.join(missing)
    )

if (SUBSET_ROOT / '.prep_done').exists():
    print(f'[ok] subset already prepared at {SUBSET_ROOT}; skipping')
else:
    cmd = [sys.executable, '-u', str(PREP_SCRIPT),
           '--out-root', str(SUBSET_ROOT),
           '--n-train', '70000', '--n-val', '10000', '--seed', '88']
    print('  cmd:', ' '.join(cmd), flush=True)
    # Stream stdout AND stderr (merged) line-by-line. subprocess.check_call
    # alone hides stderr in Colab when the child exits early - if the
    # child failed with "file not found" or argparse error, that error
    # is lost without this merging.
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            f'prepare_bdd_subset.py exited with code {rc}. See the streamed '
            f'output above for the actual error. Common causes:\n'
            f'  - dataset / labels file moved on Drive\n'
            f'  - corrupted tar/zip download\n'
            f'  - not enough free space on /content/ (~30 GB needed)'
        )
    (SUBSET_ROOT / '.prep_done').write_text('ok')

for label, d in (('images/train', SUBSET_ROOT / 'images/train2017'),
                 ('images/val',   SUBSET_ROOT / 'images/val2017'),
                 ('labels/train', SUBSET_ROOT / 'labels/train2017'),
                 ('labels/val',   SUBSET_ROOT / 'labels/val2017'),
                 ('lane_tgt/train', SUBSET_ROOT / 'lane_targets/train2017'),
                 ('lane_tgt/val',   SUBSET_ROOT / 'lane_targets/val2017')):
    n = sum(1 for _ in d.iterdir()) if d.exists() else 0
    print(f'  {label:18s} {n}')


  cmd: /usr/bin/python3 -u stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py --out-root /content/bdd_dataset --n-train 70000 --n-val 10000 --seed 88
  curve_tar   -> /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar
  labels_zip  -> /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip
  lane_tar    -> /content/drive/MyDrive/EcoCAR/datasets/lane_targets_clr_v1_polyline.tar.gz
[prep] step 1: extract curve tarball
  extracting bdd100k_clrkd_curve.tar -> /content/bdd_curve_scratch
/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/P8_train/scripts/prepare_bdd_subset.py:72: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(dest)
  done in 87.5s
  curve images: train=/content/bdd_curve_scratch/images/train (70000 jpgs), val=/content/bdd_curve_scratch/images/val (10000 jpgs)
[

### Cell 3: Launch the `clr_lane_no_square_priors` training run

batch=32, lr0=4e-4, AMP off, spatial-aug off, photometric-aug on. The
training script registers the per-epoch metrics table + Drive sync
automatically; this cell just kicks it off.


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

ROW_NAME = 'clr_lane_no_square_priors'
YAML = Path(REPO_ROOT) / 'stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane_no_square_priors.yaml'
TRAIN_SCRIPT = 'stage2/rmt_ppad_migration/P8_train/scripts/train_lane_only.py'
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

# Optional fresh-start switch - set to True to wipe /content/runs/<row>/
# and start from epoch 0. Default is resume-from-Drive (see cell 4).
FRESH = True

run_dir_local = Path('/content/runs') / ROW_NAME
if FRESH and run_dir_local.exists():
    print(f'[fresh] wiping {run_dir_local}')
    shutil.rmtree(run_dir_local, ignore_errors=True)

if not YAML.exists():
    raise FileNotFoundError(f'Missing YAML: {YAML}')

from stage2.scripts.notebook_utils import run_streaming

cmd = [
    sys.executable, '-u', TRAIN_SCRIPT,
    '--mode', 'full',
    '--name', ROW_NAME,
    '--project', '/content/runs',
    '--model-yaml', str(YAML),
    '--device', '0',
    '--workers', '8',
    '--save-period', '10',
    '--batch', '32',
    '--epochs', '250',
    '--lr0', '4e-4',
]
log = os.path.join(LOG_DIR, f'NB89_{ROW_NAME}.log')
print(f'=== launching {ROW_NAME} (batch=32, lr0=4e-4, epochs=250) ===\n', flush=True)
rc = run_streaming(cmd, log_path=log, check=False)
print(f'training rc={rc}; log -> {log}')


流式输出内容被截断，只能显示最后 5000 行内容。
      1/250      49.9G      29.31          0      1.908  6.943e-05      61.73     0.5062     0.5111        320        640:  86%|████████▌ | 1884/2188 [19:54<03:11,  1.59it/s]
      1/250      49.9G       29.3          0      1.907  6.947e-05      53.12     0.5061     0.5109        350        640:  86%|████████▌ | 1884/2188 [19:55<03:11,  1.59it/s]
      1/250      49.9G       29.3          0      1.907  6.951e-05       49.6     0.5059     0.5108        395        640:  86%|████████▌ | 1884/2188 [19:55<03:11,  1.59it/s]
      1/250      49.9G      29.29          0      1.907  6.954e-05      52.63      0.506     0.5109        364        640:  86%|████████▌ | 1884/2188 [19:56<03:11,  1.59it/s]
      1/250      49.9G      29.29          0      1.906  6.958e-05      61.65     0.5059     0.5108        323        640:  86%|████████▌ | 1884/2188 [19:56<03:11,  1.59it/s]
      1/250      49.9G      29.29          0      1.906  6.962e-05      50.04      0.506     0.510

### Cell 4 (optional): Resume from a previous Drive-saved checkpoint

If Colab disconnected mid-training, the per-epoch Drive sync left
`last.pt` and `best.pt` at
`/content/drive/MyDrive/EcoCAR/training_runs/checkpoints/<row>/`. This
cell pulls that snapshot back into `/content/runs/<row>/weights/` so
re-running cell 3 with `FRESH = False` will pick up at the saved
epoch.


In [ ]:
import shutil
from pathlib import Path

drive_ckpt = Path('/content/drive/MyDrive/EcoCAR/training_runs/checkpoints') / ROW_NAME
local_weights = Path('/content/runs') / ROW_NAME / 'weights'
local_weights.mkdir(parents=True, exist_ok=True)

restored = []
for pt_name in ('last.pt', 'best.pt'):
    src = drive_ckpt / pt_name
    if src.exists():
        shutil.copy2(src, local_weights / pt_name)
        restored.append(pt_name)
results_csv = drive_ckpt / 'results.csv'
if results_csv.exists():
    shutil.copy2(results_csv, local_weights.parent / 'results.csv')
    restored.append('results.csv')

print(f'[restore] from {drive_ckpt}:')
for name in restored:
    p = local_weights / name if name.endswith('.pt') else local_weights.parent / name
    print(f'  {name}  ({p.stat().st_size / 1e6:.1f} MB)')
if not restored:
    print('  (nothing to restore - drive dir empty)')
print('\nNow re-run cell 3 with FRESH = False to resume training.')
